# DeepFER — Facial Emotion Recognition Using Deep Learning

**Objective:** Build and evaluate a seven-class facial-expression classifier using a custom CNN and transfer learning.

### Classes
`angry`, `disgust`, `fear`, `happy`, `neutral`, `sad`, `surprise`

### Pipeline
Dataset → exploratory analysis → augmentation → class weighting → CNN baseline → MobileNetV2 transfer learning → fine-tuning → evaluation → model export → prediction/demo.

> **Important:** This system predicts visual expression labels. It should not be interpreted as a diagnostic tool or as a definitive measurement of a person's internal emotional or mental state.

In [ ]:
# Colab setup
!pip -q install -U tensorflow scikit-learn seaborn opencv-python

import os, zipfile, shutil, random, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

## 1. Dataset setup

Upload `Face Emotion Recognition Dataset(1).zip` to Colab if it is not already available. The notebook searches common locations and extracts only once.

In [ ]:
# Set this path if you upload the ZIP manually.
DATASET_ZIP = "/content/Face Emotion Recognition Dataset(1).zip"

# If running locally, change the path above.
EXTRACT_DIR = "/content/deepfer_data"

if os.path.exists(DATASET_ZIP):
    if not os.path.exists(EXTRACT_DIR):
        with zipfile.ZipFile(DATASET_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
    print("Extracted to:", EXTRACT_DIR)
else:
    print("ZIP not found. Upload the dataset ZIP and update DATASET_ZIP.")

# The supplied archive has the effective structure:
# images/images/train/<class>/*.jpg
# images/images/validation/<class>/*.jpg
DATA_ROOT = os.path.join(EXTRACT_DIR, "images", "images")
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "validation")

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Validation exists:", os.path.exists(VAL_DIR))

In [ ]:
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

def count_images(directory):
    rows = []
    for cls in CLASS_NAMES:
        p = os.path.join(directory, cls)
        n = len([x for x in os.listdir(p) if x.lower().endswith((".jpg",".jpeg",".png"))]) if os.path.isdir(p) else 0
        rows.append((cls, n))
    return pd.DataFrame(rows, columns=["emotion", "count"])

train_counts = count_images(TRAIN_DIR)
val_counts = count_images(VAL_DIR)

print("Training images:", train_counts["count"].sum())
print("Validation images:", val_counts["count"].sum())
display(train_counts)
display(val_counts)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=train_counts, x="emotion", y="count", ax=ax)
ax.set_title("Training Class Distribution")
ax.set_xlabel("Emotion")
ax.set_ylabel("Images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=val_counts, x="emotion", y="count", ax=ax)
ax.set_title("Validation Class Distribution")
ax.set_xlabel("Emotion")
ax.set_ylabel("Images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. Data loading and augmentation

Images are resized to 96×96 RGB. Augmentation is applied only to training data. The validation pipeline is deterministic and contains no augmentation.

In [ ]:
IMG_SIZE = (96, 96)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
], name="augmentation")

normalization = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 3. Class weights

The dataset is imbalanced, with `disgust` substantially underrepresented. Class weights are calculated from the training distribution so the loss function gives additional importance to minority classes.

In [ ]:
y_train = np.concatenate([y.numpy() for _, y in train_ds.unbatch().batch(4096)], axis=0)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=y_train
)
class_weight = {i: float(w) for i, w in enumerate(weights)}

print(dict(zip(CLASS_NAMES, weights.round(3))))

## 4. Custom CNN baseline

This model establishes a baseline before transfer learning. Batch normalization and dropout are used to improve optimization and reduce overfitting.

In [ ]:
def build_custom_cnn():
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = augmentation(inputs)
    x = normalization(x)

    for filters, drop in [(32, 0.20), (64, 0.25), (128, 0.30), (256, 0.35)]:
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Dropout(drop)(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)

    return tf.keras.Model(inputs, outputs, name="DeepFER_CustomCNN")

cnn = build_custom_cnn()
cnn.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
cnn.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
]

cnn_history = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=callbacks
)

In [ ]:
def plot_history(history, title):
    h = pd.DataFrame(history.history)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(h["accuracy"], label="train_accuracy")
    ax.plot(h["val_accuracy"], label="val_accuracy")
    ax.set_title(title + " — Accuracy")
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(h["loss"], label="train_loss")
    ax.plot(h["val_loss"], label="val_loss")
    ax.set_title(title + " — Loss")
    ax.legend()
    plt.show()

plot_history(cnn_history, "Custom CNN")

## 5. Transfer learning with MobileNetV2

MobileNetV2 is used as a lightweight transfer-learning backbone. The ImageNet weights provide generic visual representations; the classification head is trained for the seven emotion classes. If the runtime cannot download pretrained weights, set `weights=None`.

In [ ]:
def build_mobilenet():
    base = tf.keras.applications.MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="DeepFER_MobileNetV2")
    return model, base

model, base = build_mobilenet()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
tl_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "deepfer_mobilenetv2_best.keras",
        monitor="val_accuracy",
        save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7
    )
]

tl_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=tl_callbacks
)

plot_history(tl_history, "MobileNetV2 Transfer Learning")

## 6. Fine-tuning

After training the new classifier head, a small number of final backbone layers are unfrozen. A low learning rate is essential to avoid destroying pretrained representations.

In [ ]:
# Unfreeze only the last portion of the backbone.
base.trainable = True
fine_tune_at = max(0, len(base.layers) - 30)

for layer in base.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight,
    callbacks=tl_callbacks
)

plot_history(fine_history, "MobileNetV2 Fine-Tuning")

## 7. Detailed evaluation

Accuracy is useful but insufficient for an imbalanced seven-class problem. Precision, recall, and F1-score are reported per class, with special attention to minority classes.

In [ ]:
def collect_predictions(model, dataset):
    y_true, y_prob = [], []
    for x, y in dataset:
        p = model.predict(x, verbose=0)
        y_true.extend(y.numpy())
        y_prob.extend(p)
    return np.array(y_true), np.array(y_prob)

y_true, y_prob = collect_predictions(model, val_ds)
y_pred = y_prob.argmax(axis=1)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=np.arange(len(CLASS_NAMES)), zero_division=0
)
metrics_df = pd.DataFrame({
    "emotion": CLASS_NAMES,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support
})
display(metrics_df)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("DeepFER Confusion Matrix")
plt.tight_layout()
plt.show()

## 8. Model comparison

Run this cell after both models have been evaluated. The comparison is descriptive rather than relying on accuracy alone.

In [ ]:
cnn_true, cnn_prob = collect_predictions(cnn, val_ds)
cnn_pred = cnn_prob.argmax(axis=1)

cnn_acc = accuracy_score(cnn_true, cnn_pred)
cnn_macro_f1 = precision_recall_fscore_support(
    cnn_true, cnn_pred, average="macro", zero_division=0
)[2]

tl_acc = accuracy_score(y_true, y_pred)
tl_macro_f1 = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)[2]

comparison = pd.DataFrame({
    "model": ["Custom CNN", "MobileNetV2 Fine-Tuned"],
    "validation_accuracy": [cnn_acc, tl_acc],
    "macro_f1": [cnn_macro_f1, tl_macro_f1]
})
display(comparison)

## 9. Save the final model

The exported Keras model can be used by the OpenCV real-time application.

In [ ]:
model.save("deepfer_mobilenetv2.keras")

# Save class names as a JSON file.
with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)

print("Saved deepfer_mobilenetv2.keras and class_names.json")

## 10. Single-image prediction

Change `IMAGE_PATH` to any facial image. The function returns the predicted class and confidence.

In [ ]:
from PIL import Image

def predict_image(image_path, model=model):
    img = Image.open(image_path).convert("RGB").resize(IMG_SIZE)
    arr = np.array(img).astype("float32")
    probs = model.predict(arr[None, ...], verbose=0)[0]
    idx = int(np.argmax(probs))

    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{CLASS_NAMES[idx]} — {probs[idx]*100:.2f}%")
    plt.show()

    return CLASS_NAMES[idx], float(probs[idx])

# Example:
# predict_image("/content/example.jpg")

## 11. Real-time deployment

The included `app.py` opens the webcam, detects faces using OpenCV, resizes each face, and runs the trained Keras model. Press **q** to quit.

```bash
pip install -r requirements.txt
python app.py
```

Keep `deepfer_mobilenetv2.keras` and `app.py` in the same folder.

## 12. Limitations and responsible use

- The supplied dataset has a strongly imbalanced class distribution.
- There is no independent test split in the supplied ZIP; validation is therefore used for evaluation.
- Facial expression is not equivalent to a person's true internal emotion.
- Lighting, pose, occlusion, demographics, camera quality, and dataset bias can affect predictions.
- Real-world deployment should include an independently collected test set and fairness/error analysis.
- Avoid using predictions for high-stakes decisions about individuals.